# 02 — Business Scenario Validation

**Project:** AdIntel AI  
**Purpose:** validate the MVP business story directly from synthetic CSV files.

Main story:

> Revenue remains stable, but viewability drops around 20% and video completion rate declines around 15–20%.

The notebook dynamically splits the available date range:
- first half = **baseline**
- second half = **decline**


## 1. Setup

In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def find_project_root(start: Path | None = None) -> Path:
    """
    Find project root by walking upward until data/synthetic exists.
    This makes the notebook runnable from:
    - project root
    - notebooks/
    - VS Code interactive working directory
    """
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / "data" / "synthetic").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root. Please run this notebook from the project root "
        "or from a folder inside the project. Expected folder: data/synthetic/"
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir    : {DATA_DIR}")

def read_csv_required(filename: str, parse_dates: list[str] | None = None) -> pd.DataFrame:
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

    df = pd.read_csv(path)

    for col in parse_dates or []:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df

def safe_divide(numerator, denominator):
    denominator = pd.Series(denominator).replace(0, np.nan)
    return numerator / denominator

def pct_change(post, base):
    if pd.isna(base) or base == 0:
        return np.nan
    return post / base - 1

def fmt_pct(x):
    return "n/a" if pd.isna(x) else f"{x:.2%}"

def fmt_num(x):
    if pd.isna(x):
        return "n/a"
    if abs(x) >= 1_000_000:
        return f"{x/1_000_000:.2f}M"
    if abs(x) >= 1_000:
        return f"{x/1_000:.2f}K"
    return f"{x:,.2f}"

def add_period_by_date(df: pd.DataFrame, date_col: str = "date") -> tuple[pd.DataFrame, pd.Timestamp]:
    """
    Split data dynamically:
    - first half of available date range = baseline
    - second half of available date range = decline
    """
    if date_col not in df.columns:
        raise KeyError(f"Column `{date_col}` not found.")

    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col], errors="coerce")

    min_date = out[date_col].min()
    max_date = out[date_col].max()

    if pd.isna(min_date) or pd.isna(max_date):
        raise ValueError(f"Column `{date_col}` does not contain valid dates.")

    midpoint = min_date + (max_date - min_date) / 2
    out["period"] = np.where(out[date_col] <= midpoint, "baseline", "decline")

    return out, midpoint

def require_columns(df: pd.DataFrame, cols: list[str], table_name: str = "dataframe"):
    missing = [col for col in cols if col not in df.columns]
    if missing:
        raise KeyError(f"{table_name} is missing required columns: {missing}")

def weighted_rate(df: pd.DataFrame, numerator_col: str, denominator_col: str) -> float:
    if numerator_col not in df.columns or denominator_col not in df.columns:
        return np.nan
    den = df[denominator_col].sum()
    if den == 0:
        return np.nan
    return df[numerator_col].sum() / den

def chart_bar(df, x_col, y_col, title, xlabel=None, ylabel=None, rotation=0):
    ax = df.plot(kind="bar", x=x_col, y=y_col, figsize=(9, 5), legend=False)
    ax.set_title(title)
    ax.set_xlabel(xlabel or x_col)
    ax.set_ylabel(ylabel or y_col)
    plt.xticks(rotation=rotation)
    plt.tight_layout()
    plt.show()

def chart_line(df, x_col, y_col, title, xlabel=None, ylabel=None):
    ax = df.plot(kind="line", x=x_col, y=y_col, figsize=(11, 5), legend=False)
    ax.set_title(title)
    ax.set_xlabel(xlabel or x_col)
    ax.set_ylabel(ylabel or y_col)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 2. Load CSV Files

In [ ]:

files = {
    "advertisers": "advertisers.csv",
    "campaigns": "campaigns.csv",
    "ad_groups": "ad_groups.csv",
    "creatives": "creatives.csv",
    "placements": "placements.csv",
    "inventory": "inventory.csv",
    "markets": "markets.csv",
    "devices": "devices.csv",
    "daily_ad_performance": "daily_ad_performance.csv",
    "video_performance": "video_performance.csv",
    "conversion_events": "conversion_events.csv",
    "billing_revenue": "billing_revenue.csv",
    "data_quality_logs": "data_quality_logs.csv",
}

date_tables = {
    "daily_ad_performance",
    "video_performance",
    "conversion_events",
    "billing_revenue",
    "inventory",
    "data_quality_logs",
}

data = {
    table: read_csv_required(filename, parse_dates=["date"] if table in date_tables else None)
    for table, filename in files.items()
}

advertisers = data["advertisers"]
campaigns = data["campaigns"]
ad_groups = data["ad_groups"]
creatives = data["creatives"]
placements = data["placements"]
inventory = data["inventory"]
markets = data["markets"]
devices = data["devices"]
perf = data["daily_ad_performance"]
video = data["video_performance"]
conversions = data["conversion_events"]
billing = data["billing_revenue"]
dq_logs = data["data_quality_logs"]

print("Loaded tables:", len(data))


perf, split_date = add_period_by_date(perf, "date")
video, _ = add_period_by_date(video, "date")
inventory, _ = add_period_by_date(inventory, "date")
dq_logs, _ = add_period_by_date(dq_logs, "date")

print(f"Dynamic split date: {split_date.date()}")
print(perf.groupby("period")["date"].agg(["min", "max", "nunique"]))


## 3. Build Enriched Tables

In [ ]:

join_cols = ["performance_id", "advertiser_id", "campaign_id", "ad_group_id", "creative_id", "placement_id", "market_id", "device_id"]

perf_enriched = (
    perf
    .merge(advertisers, on="advertiser_id", how="left")
    .merge(campaigns, on="campaign_id", how="left", suffixes=("", "_campaign"))
    .merge(ad_groups, on="ad_group_id", how="left", suffixes=("", "_ad_group"))
    .merge(creatives, on="creative_id", how="left", suffixes=("", "_creative"))
    .merge(placements, on="placement_id", how="left", suffixes=("", "_placement"))
    .merge(markets, on="market_id", how="left", suffixes=("", "_market"))
    .merge(devices, on="device_id", how="left", suffixes=("", "_device"))
)

video_enriched = video.merge(
    perf_enriched[[
        col for col in [
            "performance_id",
            "creative_id",
            "period",
            "placement_id",
            "quality_tier",
            "device_id",
            "platform",
            "is_app",
            "market_id",
            "market_name",
            "inventory_type",
            "video_duration_sec",
        ]
        if col in perf_enriched.columns
    ]].drop_duplicates(),
    on=[col for col in ["performance_id", "creative_id"] if col in video.columns and col in perf_enriched.columns],
    how="left",
    suffixes=("", "_perf")
)

# Prefer period from video if available. If merge created period_perf, keep the direct video period.
if "period_perf" in video_enriched.columns:
    video_enriched = video_enriched.drop(columns=["period_perf"])

print("Performance enriched rows:", len(perf_enriched))
print("Video enriched rows      :", len(video_enriched))


## 4. Baseline vs Decline Summary

In [ ]:

def summarize_business(df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        df.groupby("period", as_index=False)
        .agg(
            impressions=("impressions", "sum"),
            clicks=("clicks", "sum"),
            spend_usd=("spend_usd", "sum"),
            publisher_revenue_usd=("publisher_revenue_usd", "sum"),
            measurable_impressions=("measurable_impressions", "sum"),
            viewable_impressions=("viewable_impressions", "sum"),
        )
    )
    summary["ctr"] = summary["clicks"] / summary["impressions"].replace(0, np.nan)
    summary["cpm"] = summary["spend_usd"] / summary["impressions"].replace(0, np.nan) * 1000
    summary["publisher_ecpm"] = summary["publisher_revenue_usd"] / summary["impressions"].replace(0, np.nan) * 1000
    summary["viewability_rate"] = summary["viewable_impressions"] / summary["measurable_impressions"].replace(0, np.nan)
    return summary

perf_summary = summarize_business(perf_enriched)
perf_summary


In [ ]:

def compare_periods(summary: pd.DataFrame, period_col="period") -> pd.DataFrame:
    baseline = summary[summary[period_col] == "baseline"].drop(columns=[period_col]).iloc[0]
    decline = summary[summary[period_col] == "decline"].drop(columns=[period_col]).iloc[0]

    comparison = pd.DataFrame({
        "metric": baseline.index,
        "baseline": baseline.values,
        "decline": decline.values,
    })
    comparison["change_abs"] = comparison["decline"] - comparison["baseline"]
    comparison["change_pct"] = comparison.apply(lambda x: pct_change(x["decline"], x["baseline"]), axis=1)
    comparison["change_pct_fmt"] = comparison["change_pct"].map(fmt_pct)
    return comparison

business_comparison = compare_periods(perf_summary)
business_comparison


## 5. Video Completion Rate Validation

In [ ]:

require_columns(video_enriched, ["period", "video_starts", "video_completes"], "video_performance")

video_summary = (
    video_enriched.groupby("period", as_index=False)
    .agg(
        video_starts=("video_starts", "sum"),
        video_completes=("video_completes", "sum"),
    )
)

video_summary["vcr"] = video_summary["video_completes"] / video_summary["video_starts"].replace(0, np.nan)
video_comparison = compare_periods(video_summary)

display(video_summary)
video_comparison


## 6. Low-Quality Placement Share

In [ ]:

if "quality_tier" not in perf_enriched.columns:
    raise KeyError("quality_tier column is required in placements.csv for this analysis.")

quality_summary = (
    perf_enriched.groupby(["period", "quality_tier"], dropna=False, as_index=False)
    .agg(
        impressions=("impressions", "sum"),
        measurable_impressions=("measurable_impressions", "sum"),
        viewable_impressions=("viewable_impressions", "sum"),
        publisher_revenue_usd=("publisher_revenue_usd", "sum"),
    )
)

quality_summary["impression_share"] = quality_summary.groupby("period")["impressions"].transform(lambda x: x / x.sum())
quality_summary["viewability_rate"] = quality_summary["viewable_impressions"] / quality_summary["measurable_impressions"].replace(0, np.nan)

quality_summary.sort_values(["period", "impression_share"], ascending=[True, False])


In [ ]:

quality_pivot = quality_summary.pivot(index="quality_tier", columns="period", values="impression_share").reset_index()
quality_pivot["share_change"] = quality_pivot.get("decline", np.nan) - quality_pivot.get("baseline", np.nan)
quality_pivot.sort_values("share_change", ascending=False)


## 7. Mobile Web vs App

In [ ]:

platform_col = "platform" if "platform" in perf_enriched.columns else None
if platform_col is None:
    raise KeyError("platform column is required in devices.csv for this analysis.")

platform_summary = (
    perf_enriched.groupby(["period", platform_col], dropna=False, as_index=False)
    .agg(
        impressions=("impressions", "sum"),
        measurable_impressions=("measurable_impressions", "sum"),
        viewable_impressions=("viewable_impressions", "sum"),
    )
)

platform_summary["impression_share"] = platform_summary.groupby("period")["impressions"].transform(lambda x: x / x.sum())
platform_summary["viewability_rate"] = platform_summary["viewable_impressions"] / platform_summary["measurable_impressions"].replace(0, np.nan)

platform_summary.sort_values(["period", "impression_share"], ascending=[True, False])


## 8. Creative Duration and VCR

In [ ]:

if "video_duration_sec" not in video_enriched.columns:
    raise KeyError("video_duration_sec column is required in creatives.csv for this analysis.")

video_enriched["duration_bucket"] = pd.cut(
    video_enriched["video_duration_sec"],
    bins=[0, 10, 15, 30, np.inf],
    labels=["<=10s", "11-15s", "16-30s", ">30s"],
    include_lowest=True,
)

duration_summary = (
    video_enriched.groupby(["period", "duration_bucket"], observed=False, as_index=False)
    .agg(
        video_starts=("video_starts", "sum"),
        video_completes=("video_completes", "sum"),
    )
)

duration_summary["vcr"] = duration_summary["video_completes"] / duration_summary["video_starts"].replace(0, np.nan)
duration_summary["start_share"] = duration_summary.groupby("period")["video_starts"].transform(lambda x: x / x.sum())

duration_summary


## 9. Data Quality Check

In [ ]:

dq_summary = (
    dq_logs.groupby(["period"], as_index=False)
    .agg(
        dq_logs=("dq_log_id", "count") if "dq_log_id" in dq_logs.columns else ("date", "count"),
        estimated_affected_rows=("estimated_affected_rows", "sum") if "estimated_affected_rows" in dq_logs.columns else ("date", "count"),
    )
)

total_perf_rows = len(perf)
dq_total_affected = dq_summary["estimated_affected_rows"].sum() if "estimated_affected_rows" in dq_summary.columns else np.nan
dq_affected_ratio = dq_total_affected / total_perf_rows if total_perf_rows else np.nan

display(dq_summary)
print(f"Estimated affected rows / performance rows: {fmt_pct(dq_affected_ratio)}")


## 10. Scenario Validation Scorecard

In [ ]:

def get_change(comparison_df, metric):
    values = comparison_df.loc[comparison_df["metric"] == metric, "change_pct"]
    return values.iloc[0] if len(values) else np.nan

revenue_change = get_change(business_comparison, "publisher_revenue_usd")
impression_change = get_change(business_comparison, "impressions")
viewability_change = get_change(business_comparison, "viewability_rate")
vcr_change = get_change(video_comparison, "vcr")

low_quality_rows = quality_pivot[quality_pivot["quality_tier"].astype(str).str.lower().str.contains("low", na=False)]
low_quality_share_change = low_quality_rows["share_change"].sum() if len(low_quality_rows) else np.nan

scorecard = pd.DataFrame([
    {
        "scenario_check": "Revenue stable or slightly up",
        "observed_change": fmt_pct(revenue_change),
        "status": "PASS" if -0.15 <= revenue_change <= 0.20 else "CHECK",
    },
    {
        "scenario_check": "Impressions increase",
        "observed_change": fmt_pct(impression_change),
        "status": "PASS" if impression_change > 0 else "CHECK",
    },
    {
        "scenario_check": "Viewability drops around 20%",
        "observed_change": fmt_pct(viewability_change),
        "status": "PASS" if -0.30 <= viewability_change <= -0.10 else "CHECK",
    },
    {
        "scenario_check": "VCR drops around 15-20%",
        "observed_change": fmt_pct(vcr_change),
        "status": "PASS" if -0.30 <= vcr_change <= -0.08 else "CHECK",
    },
    {
        "scenario_check": "Low-quality placement share increases",
        "observed_change": fmt_pct(low_quality_share_change),
        "status": "PASS" if low_quality_share_change > 0 else "CHECK",
    },
    {
        "scenario_check": "Data quality issue exists but is small",
        "observed_change": fmt_pct(dq_affected_ratio),
        "status": "PASS" if dq_affected_ratio < 0.05 else "CHECK",
    },
])

scorecard


## 11. Charts

In [ ]:

daily = summarize_business(perf_enriched.groupby(["date", "period"], as_index=False).sum(numeric_only=True))
# Rebuild cleaner daily directly to avoid accidental grouped period duplication.
daily = (
    perf_enriched.groupby("date", as_index=False)
    .agg(
        impressions=("impressions", "sum"),
        publisher_revenue_usd=("publisher_revenue_usd", "sum"),
        measurable_impressions=("measurable_impressions", "sum"),
        viewable_impressions=("viewable_impressions", "sum"),
    )
)
daily["viewability_rate"] = daily["viewable_impressions"] / daily["measurable_impressions"].replace(0, np.nan)

chart_line(daily, "date", "publisher_revenue_usd", "Daily Publisher Revenue", ylabel="Publisher Revenue USD")
chart_line(daily, "date", "impressions", "Daily Impressions", ylabel="Impressions")
chart_line(daily, "date", "viewability_rate", "Daily Viewability Rate", ylabel="Viewability Rate")

quality_chart = quality_summary.pivot(index="quality_tier", columns="period", values="impression_share")
quality_chart.plot(kind="bar", figsize=(9, 5))
plt.title("Impression Share by Placement Quality Tier")
plt.xlabel("Placement Quality Tier")
plt.ylabel("Impression Share")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

duration_chart = duration_summary.pivot(index="duration_bucket", columns="period", values="vcr")
duration_chart.plot(kind="bar", figsize=(9, 5))
plt.title("VCR by Creative Duration Bucket")
plt.xlabel("Creative Duration Bucket")
plt.ylabel("Video Completion Rate")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 12. Portfolio Interpretation

Based on the scorecard above, the expected MVP story is validated when:

- revenue is stable or slightly up,
- impressions grow,
- viewability declines materially,
- VCR declines materially,
- low-quality placement share increases,
- data quality issue exists but remains too small to explain the overall performance decline.

This supports the narrative that the problem is more likely a **quality mix shift** than a pure revenue, tracking, or data pipeline issue.
